# Chess Engine with PyTorch

## Imports

In [2]:
pip install chess

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/6.1 MB ? eta -:--:--
     ---------- ----------------------------- 1.6/6.1 MB 7.6 MB/s eta 0:00:01
     -------------------- ------------------- 3.1/6.1 MB 8.8 MB/s eta 0:00:01
     --------------------------- ------------ 4.2/6.1 MB 8.4 MB/s eta 0:00:01
     ---------------------------------------- 6.1/6.1 MB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147814 sha256=4463f55880c7fc25ba5c2d14b784eb32efc05d070df612e6ab921b9d321f1306
  Stored in directory: c:\users\siegk\appdata\local\pip\cache\wheels\83\1f\4e\8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import numpy as np # type: ignore
import time
import torch
import torch.nn as nn # type: ignore
import torch.optim as optim # type: ignore
from torch.utils.data import DataLoader # type: ignore
from chess import pgn # type: ignore
from tqdm import tqdm # type: ignore

# Data preprocessing

## Load data

In [2]:
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games

files = [file for file in os.listdir("../../data/png") if file.endswith(".pgn")]
LIMIT_OF_FILES = min(len(files), 1)
games = []
i = 1
for file in tqdm(files):
    games.extend(load_pgn(f"D:/Chess/chess-engine-main/chess-engine-main/data/png/{file}"))
    if i >= LIMIT_OF_FILES:
        break
    i += 1

  0%|          | 0/9 [08:02<?, ?it/s]


In [3]:
print(f"GAMES PARSED: {len(games)}")

GAMES PARSED: 259278


In [4]:
len(games[:50000])

50000

## Convert data into tensors

In [5]:
from auxiliary_func import create_input_for_nn, encode_moves, save_training_artifacts

In [6]:
X, y = create_input_for_nn(games[:50000])

print(f"NUMBER OF SAMPLES: {len(y)}")

NUMBER OF SAMPLES: 4370053


In [7]:
X = X[0:2500000]
y = y[0:2500000]

In [8]:
y, move_to_int = encode_moves(y)
num_classes = len(move_to_int)

In [9]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Preliminary actions

In [10]:
from dataset import ChessDataset
from model import ChessModel

In [11]:
# Create Dataset and DataLoader
dataset = ChessDataset(X, y)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

# Model Initialization
model = ChessModel(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

Using device: cuda


# Training

In [12]:
num_epochs = 50
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
        optimizer.zero_grad()

        outputs = model(inputs)  # Raw logits

        # Compute loss
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f'Epoch {epoch + 1 + 50}/{num_epochs + 1 + 50}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s')

100%|██████████| 39063/39063 [02:35<00:00, 251.40it/s]


Epoch 51/101, Loss: 3.6143, Time: 2m35s


100%|██████████| 39063/39063 [02:29<00:00, 260.67it/s]


Epoch 52/101, Loss: 2.7266, Time: 2m29s


100%|██████████| 39063/39063 [02:38<00:00, 245.68it/s]


...

Epoch 98/101, Loss: 1.5607, Time: 2m32s


100%|██████████| 39063/39063 [02:31<00:00, 257.97it/s]


Epoch 99/101, Loss: 1.5560, Time: 2m31s


100%|██████████| 39063/39063 [02:30<00:00, 260.15it/s]


Epoch 100/101, Loss: 1.5513, Time: 2m30s


# Save the model and mapping

In [13]:
# Save the model and move map with a custom base name plus the training date
saved_artifacts = save_training_artifacts(
    model=model,
    move_to_int=move_to_int,
    model_name="portfolio_test",
    models_dir="../../models",
)
print(f"Model saved to: {saved_artifacts['model_path']}")
print(f"Move map saved to: {saved_artifacts['map_path']}")

In [14]:
print(f"Training date: {saved_artifacts['trained_on']}")
print(f"Artifact prefix: {saved_artifacts['artifact_prefix']}")